In [1]:
import os
import json
import random

import cv2
import numpy as np

from PIL import Image
from tqdm import tqdm
from shapely import wkt

In [2]:
# -----------------------------
# Dataset Paths
# -----------------------------

DATASET_DIR = r"D:\archive (1)"

TRAIN_IMAGES = os.path.join(
    DATASET_DIR,
    "train",
    "train",
    "images"
)

TRAIN_LABELS = os.path.join(
    DATASET_DIR,
    "train",
    "train",
    "labels"
)

OUTPUT_DIR = r"D:\Processed_xView2_Dataset"

In [3]:
print("Images :", os.path.exists(TRAIN_IMAGES))
print("Labels :", os.path.exists(TRAIN_LABELS))
print("Output :", OUTPUT_DIR)

Images : True
Labels : True
Output : D:\Processed_xView2_Dataset


In [4]:
# -----------------------------
# Constants
# -----------------------------

IMAGE_SIZE = 224

CLASSES = [
    "no_damage",
    "minor_damage",
    "major_damage",
    "destroyed"
]

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [5]:
# -----------------------------
# Create Output Folders
# -----------------------------

for split in ["train", "val"]:
    for cls in CLASSES:
        os.makedirs(
            os.path.join(OUTPUT_DIR, split, cls),
            exist_ok=True
        )

print("Output folders created successfully!")

Output folders created successfully!


In [6]:
print(os.listdir(OUTPUT_DIR))

['train', 'val']


In [7]:
# -----------------------------
# Load Scene IDs
# -----------------------------

image_files = [
    f for f in os.listdir(TRAIN_IMAGES)
    if f.endswith("_pre_disaster.png")
]

scene_ids = sorted([
    f.replace("_pre_disaster.png", "")
    for f in image_files
])

print(f"Total scenes found: {len(scene_ids)}")

Total scenes found: 2799


In [8]:
# -----------------------------
# Scene-wise Train/Validation Split
# -----------------------------

random.shuffle(scene_ids)

split_index = int(0.8 * len(scene_ids))

train_scenes = set(scene_ids[:split_index])
val_scenes = set(scene_ids[split_index:])

print(f"Training scenes   : {len(train_scenes)}")
print(f"Validation scenes : {len(val_scenes)}")

Training scenes   : 2239
Validation scenes : 560


In [9]:
# -----------------------------
# Crop a Building from Image
# -----------------------------

def crop_building(image, polygon, padding=10):

    pts = np.array(polygon.exterior.coords, dtype=np.int32)

    x, y, w, h = cv2.boundingRect(pts)

    # Ignore very small buildings
    if w < 20 or h < 20:
        return None

    # Clip bounding box to image boundaries
    H, W = image.shape[:2]

    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(W, x + w)
    y2 = min(H, y + h)

    crop = image[y1:y2, x1:x2]

    if crop.size == 0:
        return None

    # Shift polygon coordinates
    shifted = pts.copy()
    shifted[:, 0] -= x1
    shifted[:, 1] -= y1

    # Create mask
    mask = np.zeros(crop.shape[:2], dtype=np.uint8)

    cv2.fillPoly(mask, [shifted], 255)

    # Apply mask
    result = cv2.bitwise_and(crop, crop, mask=mask)

    # Remove black borders
    ys, xs = np.where(mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    xmin = max(xs.min() - padding, 0)
    xmax = min(xs.max() + padding, result.shape[1])

    ymin = max(ys.min() - padding, 0)
    ymax = min(ys.max() + padding, result.shape[0])

    result = result[ymin:ymax, xmin:xmax]

    return result

In [10]:
# -----------------------------
# Process Building
# -----------------------------

def process_building(pre_img, post_img, polygon):

    # Crop from pre-disaster image
    pre_crop = crop_building(pre_img, polygon)

    # Crop from post-disaster image
    post_crop = crop_building(post_img, polygon)

    # Skip if either crop failed
    if pre_crop is None or post_crop is None:
        return None

    # Skip extremely small crops
    if pre_crop.shape[0] < 5 or pre_crop.shape[1] < 5:
        return None

    if post_crop.shape[0] < 5 or post_crop.shape[1] < 5:
        return None

    # Resize both images
    pre_crop = cv2.resize(pre_crop, (IMAGE_SIZE, IMAGE_SIZE))
    post_crop = cv2.resize(post_crop, (IMAGE_SIZE, IMAGE_SIZE))

    # Concatenate horizontally
    combined = np.concatenate((pre_crop, post_crop), axis=1)

    return combined

In [11]:
# -----------------------------
# Main Preprocessing Loop
# -----------------------------

class_counts = {cls: 0 for cls in CLASSES}

processed = 0
skipped = 0

for scene in tqdm(scene_ids):

    pre_path = os.path.join(
        TRAIN_IMAGES,
        scene + "_pre_disaster.png"
    )

    post_path = os.path.join(
        TRAIN_IMAGES,
        scene + "_post_disaster.png"
    )

    json_path = os.path.join(
        TRAIN_LABELS,
        scene + "_post_disaster.json"
    )

    # Skip if any file is missing
    if not (
        os.path.exists(pre_path)
        and os.path.exists(post_path)
        and os.path.exists(json_path)
    ):
        skipped += 1
        continue

    # Read images
    pre_img = cv2.cvtColor(
        cv2.imread(pre_path),
        cv2.COLOR_BGR2RGB
    )

    post_img = cv2.cvtColor(
        cv2.imread(post_path),
        cv2.COLOR_BGR2RGB
    )

    # Read annotations
    with open(json_path, "r") as f:
        data = json.load(f)

    # Decide train/validation split
    split = "train" if scene in train_scenes else "val"

    # Process every building
    for building in data["features"]["xy"]:

        label = building["properties"]["subtype"]

        # Ignore unlabeled buildings
        if label == "un-classified":
            continue

        # Convert dataset labels to folder names
        label = {
            "no-damage": "no_damage",
            "minor-damage": "minor_damage",
            "major-damage": "major_damage",
            "destroyed": "destroyed"
        }[label]

        try:
            polygon = wkt.loads(building["wkt"])
        except:
            skipped += 1
            continue

        img = process_building(
            pre_img,
            post_img,
            polygon
        )

        if img is None:
            skipped += 1
            continue

        uid = building["properties"]["uid"][:8]

        filename = f"{scene}_{uid}.png"

        save_folder = os.path.join(
            OUTPUT_DIR,
            split,
            label
        )

        os.makedirs(save_folder, exist_ok=True)

        Image.fromarray(img).save(
            os.path.join(
                save_folder,
                filename
            )
        )

        class_counts[label] += 1
        processed += 1

print("\nPreprocessing Completed!\n")

100%|██████████| 2799/2799 [50:21<00:00,  1.08s/it]  


Preprocessing Completed!



In [12]:
print("=" * 40)
print("Preprocessing Summary")
print("=" * 40)

print(f"Buildings Saved : {processed}")
print(f"Buildings Skipped : {skipped}")

print()

for cls in CLASSES:
    print(f"{cls:15s}: {class_counts[cls]}")

Preprocessing Summary
Buildings Saved : 113508
Buildings Skipped : 46286

no_damage      : 84003
minor_damage   : 10934
major_damage   : 11326
destroyed      : 7245


In [13]:
from collections import defaultdict

counts = defaultdict(int)

for split in ["train", "val"]:
    print(f"\n{split.upper()}")

    total = 0

    for cls in CLASSES:
        folder = os.path.join(OUTPUT_DIR, split, cls)

        n = len(os.listdir(folder))

        counts[(split, cls)] = n

        total += n

        print(f"{cls:15s}: {n}")

    print(f"Total: {total}")


TRAIN
no_damage      : 68158
minor_damage   : 8536
major_damage   : 9164
destroyed      : 5467
Total: 91325

VAL
no_damage      : 15845
minor_damage   : 2398
major_damage   : 2162
destroyed      : 1778
Total: 22183
